# Challenge 3: Music Genre Classification

## Huấn luyện mô hình sau Feature Engineering

Các mô hình được huấn luyện trên tập dữ liệu đã được xử lý đầy đủ, bao gồm:
- Làm sạch dữ liệu và xử lý giá trị thiếu,
- Tạo thêm các đặc trưng mới (feature engineering),
- Chuẩn hoá các biến liên tục (scaling).

Mục tiêu là đánh giá hiệu năng mô hình sau khi bổ sung đặc trưng nâng cao và tối ưu hoá dữ liệu đầu vào.

### Khai báo thư viện

In [1]:
import pandas as pd
import numpy as np
import os, sys, random
from IPython import display

from sklearn.model_selection import train_test_split, StratifiedKFold, KFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# ==== Ignore warnings ====
import warnings
warnings.filterwarnings("ignore")

### Tham số thực nghiệm

In [2]:
params = {}

# ===== Thư mục thí nghiệm =====
params["exps_dir"]  = "../exps"
params["exp_name"]  = "music_genre_baseline"
params["save_dir"]  = f'{params["exps_dir"]}/{params["exp_name"]}'

# Đường dẫn dữ liệu đã preprocess (after FE)
params["data_path"] = f'{params["exps_dir"]}/data/train_fe_pre.xlsx'
params["test_path"] = f'{params["exps_dir"]}/data/test_fe_pre.xlsx'

params["k_fold"] = 10
params["random_state"] = 42

os.makedirs(params["save_dir"], exist_ok=True)

random.seed(params["random_state"])
np.random.seed(params["random_state"])
os.environ["PYTHONHASHSEED"] = str(params["random_state"])

print("Save directory:", params["save_dir"])
print("Train path    :", params["data_path"])
print("Test path     :", params["test_path"])

Save directory: ../exps/music_genre_baseline
Train path    : ../exps/data/train_fe_pre.xlsx
Test path     : ../exps/data/test_fe_pre.xlsx


### Nạp dữ liệu

In [3]:
df_train = pd.read_excel(params["data_path"])
df_test  = pd.read_excel(params["test_path"])

In [4]:
df_train.head()

,Popularity,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,...,mood_balance,acoustic_instru,tempo_medium,tempo_fast,is_major,key_norm,tempo_sq,loudness_sq,valence_sq,Class
0,0.363636,0.295244,0.535438,0.8,0.806485,0,0.016729,0.379518,0.003935,0.096011,...,0.383093,0.000000,False,True,0,0.8,0.481286,0.027691,0.056356,9
1,0.666667,0.715946,0.746693,1.0,0.833220,1,0.069812,0.027309,0.046987,0.093970,...,0.345773,0.001313,False,True,1,1.0,0.361150,0.019257,0.148125,6
2,0.434343,0.564235,0.803763,0.6,0.819925,1,0.042252,0.000972,0.637550,0.277625,...,0.457733,0.000634,False,True,1,0.6,0.532011,0.023260,0.414479,10
3,0.111111,0.489994,0.307162,0.6,0.611251,1,0.009330,0.910643,0.021385,0.293950,...,0.662426,0.019932,False,True,1,0.6,0.621825,0.135578,0.257827,2
4,0.474747,0.543792,0.776730,0.5,0.844094,0,0.242895,0.183735,0.003935,0.203143,...,0.463953,0.000000,False,False,0,0.5,0.148159,0.016263,0.393831,5


In [5]:
df_test.head()

,Popularity,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,...,mood_intensity,mood_balance,acoustic_instru,tempo_medium,tempo_fast,is_major,key_norm,tempo_sq,loudness_sq,valence_sq
0,0.434343,0.679363,0.669600,0.1,0.795733,0,0.076783,0.076004,3.534040e-02,0.188858,...,0.449825,0.533503,2.749223e-03,False,False,0,0.1,0.154523,0.031514,0.414479
1,0.131313,0.431892,0.776730,0.1,0.786628,1,0.008686,0.389558,9.267068e-01,0.284767,...,0.431294,0.410800,3.694918e-01,False,True,1,0.1,0.553550,0.034944,0.283167
2,0.797980,0.641704,0.290141,0.1,0.711484,1,0.007292,0.875502,3.934743e-03,0.104173,...,0.091647,0.557252,0.000000e+00,True,False,1,0.1,0.213144,0.070016,0.090911
3,0.515152,0.452335,0.825789,0.6,0.856057,1,0.018445,0.000800,1.004017e-08,0.115396,...,0.599989,0.474696,8.305218e-10,True,False,1,0.6,0.178744,0.013262,0.485222
4,0.222222,0.725629,0.728672,0.0,0.812975,0,0.279357,0.147590,3.934743e-03,0.056423,...,0.620485,0.596268,0.000000e+00,False,False,0,0.0,0.104602,0.025503,0.666399


In [6]:
# Tách X, y
y = df_train["Class"].copy()
X = df_train.drop(columns=["Class"]).copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

# Split có stratify (bắt buộc khi classification)
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train X:", X_train.shape)
print("Valid X:", X_valid.shape)
print("Train y:", y_train.shape)
print("Valid y:", y_valid.shape)

X shape: (14396, 31)
y shape: (14396,)
Train X: (11516, 31)
Valid X: (2880, 31)
Train y: (11516,)
Valid y: (2880,)


In [7]:
# Dùng StratifiedKFold 
from sklearn.model_selection import StratifiedKFold

kfold = StratifiedKFold(
    n_splits=params["k_fold"],
    shuffle=True,
    random_state=params["random_state"]
)

print(f"Total rows in X_train: {len(X_train)}\n")

for fold, (train_idx, valid_idx) in enumerate(kfold.split(X_train, y_train)):
    print(f"---- Fold {fold} ----")
    print(f"Train size: {len(train_idx)}")
    print(f"Valid size: {len(valid_idx)}")
    print(f"Train idx sample: {train_idx[:10]}")
    print(f"Valid idx sample: {valid_idx[:10]}")
    print()

Total rows in X_train: 11516

---- Fold 0 ----
Train size: 10364
Valid size: 1152
Train idx sample: [0 1 2 3 4 5 6 7 8 9]
Valid idx sample: [ 25  60  73  89 100 105 123 126 138 140]

---- Fold 1 ----
Train size: 10364
Valid size: 1152
Train idx sample: [ 0  1  2  4  5  6  7  8  9 10]
Valid idx sample: [  3  27  46  64  77  83  86  92  93 107]

---- Fold 2 ----
Train size: 10364
Valid size: 1152
Train idx sample: [ 0  1  2  3  4  6  7  8  9 11]
Valid idx sample: [ 5 10 16 17 21 31 35 41 43 45]

---- Fold 3 ----
Train size: 10364
Valid size: 1152
Train idx sample: [ 1  2  3  4  5  6  7  8  9 10]
Valid idx sample: [ 0 13 24 38 40 50 55 56 69 70]

---- Fold 4 ----
Train size: 10364
Valid size: 1152
Train idx sample: [ 0  1  2  3  4  5  7  8  9 10]
Valid idx sample: [  6  14  29  30  63  67  68  90 110 119]

---- Fold 5 ----
Train size: 10364
Valid size: 1152
Train idx sample: [0 1 2 3 4 5 6 7 8 9]
Valid idx sample: [12 15 26 34 58 61 75 79 85 88]

---- Fold 6 ----
Train size: 10365
Valid s

### Lựa chọn mô hình mặc định

Các mô hình sẽ huấn luyện: Logistic Regression, Random Forest, Gradient Boosting, XGBoost và LightGBM.

In [8]:
models = [
    (
        "Logistic Regression",
        LogisticRegression(
            max_iter=1000,
            multi_class="multinomial",
            n_jobs=-1,
            random_state=params["random_state"]
        )
    ),
    (
        "Random Forest",
        RandomForestClassifier(
            n_estimators=400,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            random_state=params["random_state"],
            n_jobs=-1
        )
    ),
    (
        "Gradient Boosting",
        GradientBoostingClassifier(
            n_estimators=400,
            learning_rate=0.05,
            max_depth=3,
            random_state=params["random_state"]
        )
    ),
    (
        "XGBoost",
        XGBClassifier(
            n_estimators=800,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="multi:softprob",   # phân loại multi-class
            eval_metric="mlogloss",
            random_state=params["random_state"],
            n_jobs=-1
        )
    ),
    (
        "LightGBM",
        LGBMClassifier(
            n_estimators=800,
            learning_rate=0.05,
            max_depth=-1,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="multiclass",
            random_state=params["random_state"],
            n_jobs=-1,
            verbose=-1,           # tắt bớt log
            force_row_wise=True   
        )
    ),
]

### Huấn luyện và đánh giá từng mô hình

In [9]:
results = []
baseline_results = {}

for name, model in models:
    print(f"===== Model: {name} =====")

    baseline_results[name] = {
        "acc": [],
        "f1w": []
    }

    skf = StratifiedKFold(
        n_splits=params["k_fold"],
        shuffle=True,
        random_state=params["random_state"]
    )

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[valid_idx], y.iloc[valid_idx]

        # Train
        model.fit(X_tr, y_tr)

        # Predict
        y_pred = model.predict(X_val)

        # Metrics
        acc = accuracy_score(y_val, y_pred)
        f1w = f1_score(y_val, y_pred, average="weighted")

        baseline_results[name]["acc"].append(acc)
        baseline_results[name]["f1w"].append(f1w)

        print(f" Fold {fold}: ACC = {acc:.4f} | F1-weighted = {f1w:.4f}")

    mean_acc = np.mean(baseline_results[name]["acc"])
    std_acc  = np.std(baseline_results[name]["acc"])
    mean_f1w = np.mean(baseline_results[name]["f1w"])
    std_f1w  = np.std(baseline_results[name]["f1w"])

    print(f"--> Mean ACC: {mean_acc:.4f} ± {std_acc:.4f}")
    print(f"--> Mean F1w: {mean_f1w:.4f} ± {std_f1w:.4f}\n")

    results.append([
        name,
        mean_acc,
        mean_f1w
    ])

===== Model: Logistic Regression =====
 Fold 0: ACC = 0.4979 | F1-weighted = 0.4695
 Fold 1: ACC = 0.5076 | F1-weighted = 0.4766
 Fold 2: ACC = 0.4924 | F1-weighted = 0.4623
 Fold 3: ACC = 0.4993 | F1-weighted = 0.4653
 Fold 4: ACC = 0.4903 | F1-weighted = 0.4621
 Fold 5: ACC = 0.4813 | F1-weighted = 0.4516
 Fold 6: ACC = 0.5142 | F1-weighted = 0.4836
 Fold 7: ACC = 0.4969 | F1-weighted = 0.4677
 Fold 8: ACC = 0.4948 | F1-weighted = 0.4626
 Fold 9: ACC = 0.4823 | F1-weighted = 0.4530
--> Mean ACC: 0.4957 ± 0.0097
--> Mean F1w: 0.4654 ± 0.0092

===== Model: Random Forest =====
 Fold 0: ACC = 0.5056 | F1-weighted = 0.4916
 Fold 1: ACC = 0.4938 | F1-weighted = 0.4835
 Fold 2: ACC = 0.5215 | F1-weighted = 0.5089
 Fold 3: ACC = 0.5049 | F1-weighted = 0.4914
 Fold 4: ACC = 0.5000 | F1-weighted = 0.4871
 Fold 5: ACC = 0.5160 | F1-weighted = 0.5016
 Fold 6: ACC = 0.5316 | F1-weighted = 0.5172
 Fold 7: ACC = 0.5031 | F1-weighted = 0.4874
 Fold 8: ACC = 0.4955 | F1-weighted = 0.4815
 Fold 9: ACC

**Nhận xét kết quả thực nghiệm các mô hình**

- Logistic Regression đạt Accuracy trung bình 49.57% và F1-weighted 46.54%.
→ Hiệu suất thấp, dao động giữa các fold nhỏ, cho thấy mô hình tuyến tính không phù hợp để mô tả các quan hệ phức tạp và phi tuyến giữa các đặc trưng âm thanh.

- Random Forest cải thiện nhẹ so với Logistic Regression, đạt Accuracy 50.56% và F1-weighted 49.23%.
→ Mô hình cây quyết định học tốt hơn các quan hệ phi tuyến, nhưng mức cải thiện chưa lớn; hiệu suất vẫn chưa thật sự cao.

- Gradient Boosting đạt hiệu suất cao nhất trong tất cả mô hình, với Accuracy 53.93% và F1-weighted 51.90%.
→ Mô hình khai thác mạnh các tương tác phi tuyến và đặc trưng đã tạo (FE).
→ Độ dao động thấp giữa các fold (±1.34%) cho thấy sự ổn định và khả năng tổng quát hóa tốt.

- XGBoost đạt Accuracy 51.19%, F1-weighted 50.24%, tương đương Random Forest.
→ Mặc dù mạnh hơn cây đơn thuần, nhưng trong cấu hình mặc định vẫn chưa vượt qua Gradient Boosting.

- LightGBM đạt Accuracy 50.74% và F1-weighted 49.84%, kết quả khá gần XGBoost và Random Forest.
→ Tốc độ huấn luyện nhanh, ổn định nhưng hiệu suất vẫn chưa bằng Gradient Boosting.

Kết luận: Gradient Boosting là mô hình hoạt động tốt nhất sau Feature Engineering, với Accuracy ~54%, vượt các mô hình khác từ 2–4%.